## Notebook for Supervised Fine Tuning Details
<a target="_blank" href="https://colab.research.google.com/github/santoshborse/external_work/blob/test/sft-step-by-step.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
print(f"{tokenizer.chat_template}")

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
chat_template = """
    {%- for message in messages %}
        {%- if message['role'] == 'user' %}
            {{- '<|user|>\n' + message['content'] + '\n' }}
        {% elif message['role'] == 'assistant' %}
            {{- '<|assistant|>\n'  + message['content'] + eos_token }}
        {% endif %}
    {%- endfor -%}
    """
tokenizer.chat_template = chat_template

In [ ]:
messages = [
    {"role": "user", "content": "write kubectl command to delete the pod px"},
    {"role": "assistant", "content": "kubectl delete pod px"}
]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(formatted_prompt)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/pythia-14m",
    revision="main",
    trust_remote_code="False",
    use_fast=True,
)

chat_template = """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>
' + message['content'] + '
' }}{% elif message['role'] == 'user' %}{{ '<|user|>
' + message['content'] + '
' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>
'  + message['content'] + eos_token + '
' }}{% else %}{{ '<|assistant|>
'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>
' }}{% endif %}{% endfor %}"""
tokenizer.chat_template = chat_template

In [ ]:
messages = [
    {"role": "user", "content": "write kubectl command to delete the pod px"},
    {"role": "assistant", "content": "kubectl delete pod px"}
]

In [ ]:
formatted_prompt = tokenizer.apply_chat_template(conversation=messages,
        tokenize=False,
        return_tensors="pt",
        padding=False,
        truncation=True,
        max_length=1024,
        add_generation_prompt=False)
print(f"{repr(formatted_prompt)}")

In [ ]:
inputs = tokenizer(formatted_prompt, return_tensors="pt")
print(f"Token Count: {len(inputs.input_ids[0])}")
print(f"Tokens: {inputs.input_ids[0]}")

In [ ]:
def visualize_token(tokens: list[int], tokenizer):
    tok_list = []
    for i,token in enumerate(tokens):
        decoded_token = token
        if token != -100:
            decoded_token = tokenizer.decode(token)
        print(f"{token:<5} -> {repr(decoded_token):<15}           ", end='' )
        tok_list.append([f"{token:<5}", f"{repr(decoded_token):<15}"])
        if i % 3 == 1:
            print("\r\n")
visualize_token(inputs.input_ids.tolist()[0], tokenizer)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

model_name ="EleutherAI/pythia-14m"
model = AutoModelForCausalLM.from_pretrained(
                model_name,
                revision=None,
                torch_dtype=torch.bfloat16,
                attn_implementation="eager"
)

In [ ]:
outputs = model(**inputs, use_cache=False)
logits = outputs.logits
print(f"{logits.shape = }")

In [ ]:
labels = torch.tensor([[ -100,  -100, -100, -100, -100,  -100,  -100, -100, -100, -100, -100, -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100, -100,  -100,  -100,  -100,  -100,    
          76,   538,   646, 77, 11352, 7360, 268, 89, 0]])
print(f"{logits.shape = }") 
print(f"{labels.shape = }")
# logits.shape = torch.Size([1, 34, 50304])
# labels.shape = torch.Size([1, 34])

In [ ]:
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()
print(f"{shift_logits.shape = }") 
print(f"{shift_labels.shape = }") 

# shift_logits.shape = torch.Size([1, 33, 50304])
# shift_labels.shape = torch.Size([1, 33])

In [ ]:
embedding_size = 50304
shift_logits = shift_logits.view(-1, embedding_size)
shift_labels = shift_labels.view(-1)
print(f"{shift_logits.shape = }") 
print(f"{shift_labels.shape = }") 

# shift_logits.shape = torch.Size([33, 50304])
# shift_labels.shape = torch.Size([33])

In [ ]:
loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
loss = loss_fct(shift_logits, shift_labels)
loss

# tensor(70.5000, dtype=torch.bfloat16, grad_fn=<NllLossBackward0>)